# GAN_SQLi — Branch C: C-SeqGAN-Minimal

Notebook này clone repo, cài dependency, tự tìm dataset V5.1, rồi chạy một lượt C-SeqGAN-Minimal.

Mặc định là smoke run mạnh hơn mức kiểm tra rỗng: MLE + discriminator + adversarial fast policy-gradient + generate candidates.


In [ ]:
# ===== CONFIG =====
REPO_URL = 'https://github.com/MinhBe/GAN_SQLi.git'
REPO_DIR = '/content/GAN_SQLi'

# Nếu bạn đặt dataset zip trong repo, giữ path này.
# Nếu dataset nằm nơi khác, sửa DATA_PATH hoặc DATA_URL.
DATA_PATH = f'{REPO_DIR}/SQLGAN_PostgreSQL_Boolean_CSeqGAN_V5_1_CoverageFixed.zip'
DATA_URL = ''  # optional: direct download URL to the V5.1 zip

OUT_DIR = '/content/runs_sqlgan/branch_c_cseqgan_minimal_smoke'
SMOKE_MODE = True
USE_PARALLEL_MODULES = 'auto'  # auto | on | off
CPU_THREADS = 2


In [ ]:
import os, subprocess, sys, json, pathlib, textwrap, glob, shutil
from pathlib import Path

def run(cmd, cwd=None):
    print('\n$ ' + cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not Path(REPO_DIR).exists():
    run(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print('Repo already exists:', REPO_DIR)
    run('git pull --ff-only || true', cwd=REPO_DIR)

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())


In [ ]:
# ===== INSTALL DEPENDENCIES =====
run('python -m pip install -U pip')
if Path('requirements.txt').exists():
    run('python -m pip install -r requirements.txt')
run('python -m pip install -e .')


In [ ]:
# ===== HARDWARE CHECK =====
import torch, multiprocessing
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
print('cuda device count =', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 1024**3, 2), 'GB')
print('cpu count visible =', multiprocessing.cpu_count())


In [ ]:
# ===== DATASET RESOLUTION =====
from pathlib import Path
if DATA_URL and not Path(DATA_PATH).exists():
    run(f'python - <<\'PY\'\nimport urllib.request\nurl={DATA_URL!r}\nout={DATA_PATH!r}\nprint("Downloading", url, "->", out)\nurllib.request.urlretrieve(url, out)\nPY')

if not Path(DATA_PATH).exists():
    candidates = glob.glob(f'{REPO_DIR}/**/*CSeqGAN*V5*CoverageFixed*.zip', recursive=True)
    candidates += glob.glob(f'{REPO_DIR}/**/*V5_1*.zip', recursive=True)
    if candidates:
        DATA_PATH = candidates[0]
        print('Auto-selected DATA_PATH =', DATA_PATH)

assert Path(DATA_PATH).exists(), f'Dataset zip not found: {DATA_PATH}'
print('DATA_PATH =', DATA_PATH)
print('size MB =', round(Path(DATA_PATH).stat().st_size / 1024**2, 2))


In [ ]:
# ===== BRANCH C RUN PARAMETERS =====
if SMOKE_MODE:
    params = dict(
        limit=2048,
        max_len=256,
        batch_size=160,
        mle_epochs=6,
        d_epochs=1,
        adv_epochs=1,
        adv_steps=25,
        emb_dim=128,
        hidden_dim=256,
        generate_per_ruleset=2,
        generate_ruleset_limit=100,
    )
else:
    params = dict(
        limit=None,
        max_len=256,
        batch_size=192,
        mle_epochs=16,
        d_epochs=2,
        adv_epochs=2,
        adv_steps=60,
        emb_dim=128,
        hidden_dim=256,
        generate_per_ruleset=3,
        generate_ruleset_limit=1000,
    )
print(json.dumps(params, indent=2))


In [ ]:
# ===== RUN C-SEQGAN-MINIMAL =====
limit_arg = '' if params['limit'] is None else f"--limit {params['limit']}"
cmd = f"""
python -m sqlgan_dual.experiment_c_conditioned_seqgan \
  --data {DATA_PATH} \
  --out {OUT_DIR} \
  --modules Y1_basic_boolean Y2_boolean_variation \
  {limit_arg} \
  --max-len {params['max_len']} \
  --batch-size {params['batch_size']} \
  --mle-epochs {params['mle_epochs']} \
  --d-epochs {params['d_epochs']} \
  --adv-epochs {params['adv_epochs']} \
  --adv-steps {params['adv_steps']} \
  --emb-dim {params['emb_dim']} \
  --hidden-dim {params['hidden_dim']} \
  --temperature 0.85 \
  --static-weight 0.45 \
  --generate-per-ruleset {params['generate_per_ruleset']} \
  --generate-ruleset-limit {params['generate_ruleset_limit']} \
  --parallel-modules {USE_PARALLEL_MODULES} \
  --cpu-threads {CPU_THREADS} \
  --num-workers 0
"""
run(' '.join(cmd.split()))


In [ ]:
# ===== QUICK RESULT CHECK =====
import pandas as pd, json
summary_path = Path(OUT_DIR) / 'experiment_summary.json'
print(summary_path)
print(summary_path.read_text()[:4000])
for csv in Path(OUT_DIR).glob('*/generated_candidates/generated_candidates.csv'):
    df = pd.read_csv(csv)
    print('\n', csv)
    print('rows =', len(df))
    cols = [c for c in ['reward_total','ruleset_match_score','skeleton_match_score','slot_match_score','accepted_for_next_stage'] if c in df.columns]
    print(df[cols].describe(include='all'))
    display(df.head(10))


In [ ]:
# ===== OPTIONAL: ZIP RESULTS FOR DOWNLOAD =====
zip_path = '/content/branch_c_cseqgan_minimal_results.zip'
run(f'cd {Path(OUT_DIR).parent} && zip -qr {zip_path} {Path(OUT_DIR).name}')
print(zip_path)
